<p style="text-align: right; margin: 0; line-height: 0;"><img width="100px" height="auto" src="../../.admin/img/eth_logo.png"></p>

## 05.05 Introduction to LLMs

<div class="alert alert-light">

  **Name:** (add your name here)
  
  **Matriculation Number:** (add your number here)
  
</div>



### 1. Introduction

Large Language Models (LLMs) are a class of artificial intelligence systems designed to process and generate human-like text. At their core, they transform an input sequence (a question, instruction, or statement) into an output sequence (a continuation, answer, or reformulation).

LLMs look at the words already written and guess the following token, one **token** at a time.

Early AI models were typically designed for narrow tasks, such as machine translation: turning an English sentence into its equivalent in Italian. LLMs take this same idea and expand it. Instead of focusing on one language pair or one narrow task, they learn **general patterns** of text across vast amounts of data.

By training on billions of examples, they can adapt these patterns to a wide range of outputs. For example:
- translating text into multiple languages
- generating programming code
- drafting creative stories or poems
- answering factual questions
- summarizing long articles into concise paragraphs

The breakthrough came with the Transformer architecture in 2017, which made it feasible to train very large models efficiently. Several trends aligned:
- Data availability: massive public and proprietary text datasets
- Compute power: GPUs/TPUs capable of training billions of parameters
- Research advances: self-attention, scaling laws, instruction tuning
- Industry adoption: strong results drove rapid integration into products

In just a few years, LLMs evolved from research labs into everyday applications across search, productivity tools, education, and software development.

### 2. Inner Workings: How LLMs Generate Text

At a high level, LLMs generate text by repeatedly predicting the next token, given the previous tokens: “Given everything so far, what is the most plausible next token?” Training happens step by step. First, the text is broken into tokens, usually words or subwords. Each token is then mapped to a numerical vector, called an embedding, so the model can work with numerical values instead of text. The model uses self-attention to decide which earlier tokens matter most for the current one, creating weighted shortcuts such as numbers to units, adjectives to nouns, verbs to subjects or objects, or quotes to speakers. These relationships are processed through transformer blocks, which are stacked layers combining self-attention with small neural computations. Each layer refines the token vectors, gradually improving their representation. From there, the model produces a probability distribution over possible next tokens, and a decoding strategy selects one token to add to the output. This process repeats until the model completes its response or reaches a length limit.

Every possible token (word, subword, or symbol) has a unique ID in the model’s vocabulary.

   ```
   "The" → ID 5023
   "cat" → ID 1347
   "sleeps" → ID 8094
   ```

These IDs are just lookup keys, just like dictionary entries. A model stores a relatively big table (a matrix) where each row corresponds to one token ID.

   ```
   ID 1347 ("cat") → [0.9, -0.1, 0.4, 0.7]
   ID 8094 ("sleeps") → [0.2, 0.8, 0.1, -0.5]
   ```

This is how the model turns “token ID 1347” into its embedding vector. The token ID is an index into the embedding matrix. When the model sees “cat”, it looks up row 1347 in the embedding table and pulls out the vector.

Embeddings are coordinates in a high-dimensional space and can be negative. It seems like they should only be between 0 and 1, but embeddings are different. When a model predicts the next token, it outputs probabilities, and those are indeed between 0 and 1, summing to 1. But embeddings are not probabilities; they are vectors of real numbers. They can be positive or negative. Think of embeddings like coordinates in a space. Positive and negative values let the model place words in all directions around the origin (0,0,...). For example, if one dimension captures “animal vs. object”, then one possible embedding could include “cat” = +0.8 and “car” = –0.7. Without negatives, the space would be one-sided, and the model could not represent opposites. Embeddings are stored in the same way as neural network weights. Neural nets use numbers (often normalized to the range [–1,1]) because activations and linear algebra work best when values can swing both ways.<br> To summarize, embeddings live in $\mathbb{R}^d$ and can be negative, whereas probabilities live in $[0,1]$ and sum to 1.

Self-attention is the mechanism a transformer model uses to figure out which earlier words matter most when interpreting the current word. Imagine the sentence: `The cat sat on the mat because it was tired.` When the model reaches the word “it”, self-attention asks: *which earlier words should “it” connect to?* The model gives a high weight (e.g. 0.85) to “cat” because “it” most likely refers to the cat. It does this based on earlier training. It might give a low weight (e.g. 0.10) to “mat” since that is less likely to be “tired”. Words like “the” or “on” get almost no weight (approaching 0.0). So instead of treating “it” in isolation, the model builds a weighted summary of the relevant context ("cat") and uses that to interpret “it” correctly. This weighting process happens for every token, across many layers, allowing the model to capture long-range relationships like subjects with verbs, units with numbers, or quotes with speakers.

The model repeats this process again and again, each time at a higher level of abstraction, gradually building a richer map of connections. A transformer is not just one self-attention step. It has many layers, each with its own self-attention mechanism. Think of it like stacking sheets of tracing paper: The first layer might notice local links, like “it → cat.” The next layer builds on that, combining “cat” with “sat” to form a subject-verb link. A later layer might connect “because” with the whole clause that follows. By the time information has passed across many layers, the model has refined and enriched its understanding of the relationships in the sentence: From small, nearby links to broad, long-range ones.

At the end of this whole iterative process, the final representations are turned into logits, raw scores for every possible next token, which are then converted into probabilities.

#### 2.1 Token Generation Example

Consider the following tiny model with a handful of tokens and weights. Tokens are chosen according to those weights. 

> ```
> hello ──0.4──▶ world ──0.8──▶ !
>   │              │
>   │              └─0.2──▶ .
>   │
>   └─0.6──▶ there ──0.3──▶ !
>                   │
>                   └─0.7──▶ ,
>
> , ──1.0──▶ my ──0.8──▶ friend ──1.0──▶ !
>             │
>             └─0.2──▶ enemy ──1.0──▶ !

In this example, the dictionary maps tokens to their possible next tokens with simple weights (numbers between 0 and 1). We sample according to those weights, append the chosen token, and repeat. We stop at an end token (`!` or `.`) or when we reach a defined length limit. Below, we run the token generation multiple times.

In [ ]:
import random

tiny_model = {
    "hello": {"world": 0.4, "there": 0.6},
    "world": {"!": 0.8, ".": 0.2},
    "there": {"!": 0.3, ",": 0.7},
    ",": {"my": 1.0},
    "my": {"friend": 0.8, "enemy": 0.2},
    "friend": {"!": 1.0},
    "enemy": {"!": 1.0}
}

EOS = {"!", "."}
PUNCT = {",", "!", "."}

def simple_predict(start_word: str, max_output_tokens: int = 6) -> str:
    seq = [start_word]
    for _ in range(max_output_tokens):
        prev_token = seq[-1]
        if prev_token in EOS:
            break
        options = tiny_model.get(prev_token)
        if not options:
            break
        tokens, weights = zip(*options.items())
        next_token = random.choices(tokens, weights=weights, k=1)[0]
        seq.append(next_token)

    out = "".join(token if (i == 0 or token in PUNCT) else " " + token
                   for i, token in enumerate(seq))

    return out

for _ in range(10):
    print(simple_predict("hello"))

The prediction can use more than just the last token.
We can compare two settings:

- **Window = 1:** decide the next token using only the **last** token
- **Window = 2:** decide using the **last two** tokens (a step toward context awareness)

The goal is to match singular/plural:
- “**this** red **cat**.” (singular)
- “**these** red **cats**.” (plural)

In [ ]:
import random

rules_1 = {  # decisions using ONLY the last token
    "this": {"red": 0.5, "blue": 0.5},
    "these": {"red": 0.5, "blue": 0.5},
    "red": {"cat": 0.6, "cats": 0.4},
    "blue": {"cat": 0.6, "cats": 0.4},
    "cat": {".": 1.0},
    "cats": {".": 1.0},
}

rules_2 = {  # decisions using the last TWO tokens
    ("this", "red"): {"cat": 0.9, "cats": 0.1},
    ("this", "blue"): {"cat": 0.9, "cats": 0.1},
    ("these", "red"): {"cat": 0.1, "cats": 0.9},
    ("these", "blue"): {"cat": 0.1, "cats": 0.9},
}

EOS = {"."}

def case_predict(start_word: str, window: int = 1, max_output_tokens: int = 6) -> str:
    seq = [start_word]
    for _ in range(max_output_tokens):
        prev_token = seq[-1]
        if prev_token in EOS:
            break

        options = None
        if window >= 2 and len(seq) >= 2:
            prev_pair = (seq[-2], seq[-1])
            options = rules_2.get(prev_pair)

        if options is None:
            options = rules_1.get(prev_token)

        if not options:
            break
        tokens, weights = zip(*options.items())
        next_token = random.choices(tokens, weights=weights, k=1)[0]
        seq.append(next_token)

    out = "".join(tok if (i == 0 or tok in PUNCT) else " " + tok
                   for i, tok in enumerate(seq))

    return out

First, the singular case (this + adjective → cat):

In [ ]:
print("Window = 1")
for _ in range(6):
    print(" ", case_predict("this", window=1))

print("\nWindow = 2")
for _ in range(6):
    print(" ", case_predict("this", window=2))

Next, the plural case (these + adjective → cats):

In [ ]:
print("Window = 1")
for _ in range(6):
    print(" ", case_predict("these", window=1))

print("\nWindow = 2")
for _ in range(6):
    print(" ", case_predict("these", window=2))

<div class="alert alert-block alert-info">

Now, try adjusting the `cat`/`cats` weights and see how the output changes.
</div>

### 3. Models

Modern large language models are less distinguished by their underlying architecture than by the way they are trained. What primarily shapes their performance is how much text they are exposed to, how carefully that text is curated, and how training resources are balanced between model size and data volume.

In recent years, training datasets have expanded dramatically, growing from hundreds of billions to several trillions of tokens. At the same time, researchers have learned that the quality of data can matter more than sheer size. Removing duplicates, incorporating multilingual material, and including specialized domains such as programming and mathematics often improve results more than simply adding more text.

Another important development is the idea of compute-optimal training. Studies have shown that, under a fixed budget, it can be more effective to train a smaller model on a much larger dataset than to build an extremely large model trained on limited text. Finally, the variety of training formats has increased. Models are no longer trained only on raw web text, but also on code, question-and-answer examples, mathematical reasoning, and instruction-style data. This diversity shapes how flexibly and reliably the models can respond to different tasks.

### 4. Decoding Parameters: How Probabilities Become Words

When a model is about to generate the next token, it produces a probability distribution over its vocabulary.
Decoding parameters control how we turn that distribution into actual text. Each setting influences the resulting style, creativity, and stability of the output. The examples below show how decoding parameters reshape the probability distribution for a next token. The last one, Max Tokens, is a special case as it is a hard cutoff and does not affect the weights.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

MODEL_NAME = "google/flan-t5-small"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
model.eval()

def top_tokens_from_probabilities(probabilities, k=10):
    top = torch.topk(probabilities, k)
    ids = top.indices.tolist()
    tokens = tokenizer.convert_ids_to_tokens(ids)
    values = top.values.tolist()
    return list(zip(tokens, [round(float(v), 4) for v in values]))

def apply_temperature(logits, temp: float):
    temp = max(float(temp), 1e-8)
    return logits / temp

def apply_repetition_penalty_logits(logits, penalize_token_ids, penalty: float):
    if penalty <= 1.0 or len(penalize_token_ids) == 0:
        return logits
    penalized = logits.clone()
    penalized[:, penalize_token_ids] = penalized[:, penalize_token_ids] / penalty
    return penalized

def top_k_top_p_filtering(logits, top_k: int = 0, top_p: float = 1.0, filter_value: float = -float("inf")):
    logits = logits.clone()
    # Top-k
    if top_k and top_k > 0:
        top_k = min(top_k, logits.size(-1))
        threshold = torch.topk(logits, top_k).values[..., -1, None]
        remove = logits < threshold
        logits[remove] = filter_value
    # Top-p
    if top_p and top_p < 1.0:
        sorted_logits, sorted_indices = torch.sort(logits, descending=True)
        sorted_probs = torch.softmax(sorted_logits, dim=-1)
        cum_probs = torch.cumsum(sorted_probs, dim=-1)
        remove = cum_probs > top_p
        remove[..., 0] = False  # keep at least 1
        indices_to_remove = sorted_indices[remove]
        logits[0, indices_to_remove] = filter_value
    return logits

def show_next_token_table(
    prompt: str,
    *,
    T=1.0,
    top_k=0,
    top_p=1.0,
    repetition_penalty=1.0,
    penalize_from_prompt=True,
    k=10,
    header=None
):
    encoder = tokenizer(prompt, return_tensors="pt")
    decoder_start = model.config.decoder_start_token_id
    decoder_input_ids = torch.tensor([[decoder_start]])

    with torch.no_grad():
        outputs = model(**encoder, decoder_input_ids=decoder_input_ids)
    logits = outputs.logits[:, -1, :]

    penalize_ids = set()
    if penalize_from_prompt:
        penalize_ids = set(encoder["input_ids"].view(-1).tolist())

    # Apply decoding transforms
    logits = apply_temperature(logits, T)
    logits = apply_repetition_penalty_logits(logits, list(penalize_ids), repetition_penalty)
    logits = top_k_top_p_filtering(logits, top_k=top_k, top_p=top_p)

    probabilities = torch.softmax(logits, dim=-1).squeeze(0)
    if header:
        print("\n" + header)
    print(f"Prompt: {prompt!r}")
    print(f"Settings: T={T}, top_k={top_k}, top_p={top_p}, repetition_penalty={repetition_penalty}")
    for token, p in top_tokens_from_probabilities(probabilities, k=k):
        print(f"{token:>15}  {p:.4f}")

BASE_PROMPT = "Write one short sentence about Zurich coffee culture."
REP_PROMPT = "AI AI AI AI AI. Continue the sentence naturally."

#### 4.1 Decoding-Parameter "Temperature"

- **Idea**: sets how firmly the model prefers its top choices

- **How it works** (intuitively): the model gives internal scores for each candidate token

Temperature rescales those scores before converting them to probabilities:

- **Lower `T`** (<0.5): the highest-scoring tokens dominate. Outputs feel firm and consistent, but can be repetitive
- **Medium `T`** (≈0.7): balanced. Usually a good default for precise yet flexible text
- **Higher `T`** (>1.0): probability differences between tokens are softened. The model is less firm, willing to consider more alternatives. Outputs become more varied and seemingly more random/erratic



In [ ]:
show_next_token_table(BASE_PROMPT, T=0.3, header="— Temperature low —")
show_next_token_table(BASE_PROMPT, T=1.3, header="— Temperature high —")

#### 4.2 Decoding-Parameter "Top-k"

- Idea: make a fixed-size shortlist. Keep only the *k* most likely next tokens, ignore the rest, then sample from that shortlist

Top-k removes non-fitting long-tail options with low probabilities:

- **Smaller `k` (<25)**: tighter shortlist. More stable and focused output, but can feel dull or repetitive.
- **Larger `k` (>75)**: wider shortlist. More variety and creativity, but higher chance of drifting or odd word choices

In [ ]:
show_next_token_table(BASE_PROMPT, top_k=3, header="— Top-k small —")
show_next_token_table(BASE_PROMPT, top_k=10, header="— Top-k large —")

#### 4.3 Decoding-Parameter "Top-p (nucleus sampling)"

- Idea: make a dynamic shortlist. Keep the smallest set of tokens whose probabilities add up to p

Nucleus sampling adapts to uncertainty. When the model is very confident, the shortlist stays small. When it is unsure, it grows:

- **Smaller `p`** (<0.8): very selective shortlist. Safer and more focused, but sometimes more repetitive
- **Larger `p`** (>0.9): broad shortlist. More variety and creativity, but higher risk of tangents or odd word choices

In [ ]:
show_next_token_table(BASE_PROMPT, top_p=0.5, header="— Top-p small —")
show_next_token_table(BASE_PROMPT, top_p=0.95, header="— Top-p large —")

#### 4.4 Decoding-Parameter "Repetition Penalty"

- Idea: push the model away from repeating words/phrases it just used. Internally, tokens that appeared recently are penalized before sampling

Repetition penalty reduces loops and echoes (“Zurich Zurich Zurich…”):

- **Smaller `penalty`** (≈1.0): gentle or no push against repeats. Natural reuse of key terms. Can drift into mild repetition
- **Larger `penalty`** (>1.2): stronger push against repeats. Fresher wording but risk of unnatural phrasing or dodging necessary terms

In [ ]:
show_next_token_table(REP_PROMPT, repetition_penalty=1.0, header="— Repetition penalty none —")
show_next_token_table(REP_PROMPT, repetition_penalty=1.5, header="— Repetition penalty high —")

#### 4.5 Decoding-Parameter "Max Tokens (Output Length)"

- Idea: hard cap how many new tokens the model may generate

Max tokens decides how long you keep sampling. It controls verbosity, latency, and cost:

- **Smaller `limit`** (<128): concise, fast and cheap output. Higher risk of truncation or missing details
- **Larger `limit`** (>2048): more complete answers and narratives. Slower and more verbose, with higher cost

#### 4.6 Combination of Decoding-Parameters

<div class="alert alert-block alert-info">
Use the next-token table to see how multiple parameters interact. Feel free to swap in your own prompt.
</div>

> *Hint*: this is specifically helpful for the Questions section.

In [ ]:
prompt = "Write one short sentence about Zurich coffee culture."

show_next_token_table(prompt, T=1.0, top_k=0, top_p=1.0, repetition_penalty=1.0, header="— Cross-Interaction —")


###  Literature

[01] Vaswani, A., Shazeer, N., Parmar, N., Uszkoreit, J., Jones, L., Gomez, A. N., ... & Polosukhin, I. (2017).
**Attention Is All You Need.** Advances in Neural Information Processing Systems (NeurIPS).
[Paper](https://arxiv.org/abs/1706.03762)

[02] Devlin, J., Chang, M. W., Lee, K., & Toutanova, K. (2019).
**BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding.** NAACL-HLT.
[Paper](https://arxiv.org/abs/1810.04805)

[03] Radford, A., Narasimhan, K., Salimans, T., & Sutskever, I. (2018).
**Improving Language Understanding by Generative Pre-Training (GPT-1).** OpenAI.
[Paper](https://cdn.openai.com/research-covers/language-unsupervised/language_understanding_paper.pdf)

[04] Radford, A., Wu, J., Child, R., Luan, D., Amodei, D., & Sutskever, I. (2019).
**Language Models are Unsupervised Multitask Learners (GPT-2).** OpenAI.
[Paper](https://cdn.openai.com/better-language-models/language_models_are_unsupervised_multitask_learners.pdf)

[05] Brown, T. B., Mann, B., Ryder, N., Subbiah, M., Kaplan, J., Dhariwal, P., ... & Amodei, D. (2020).
**Language Models are Few-Shot Learners (GPT-3).** Advances in Neural Information Processing Systems (NeurIPS).
[Paper](https://arxiv.org/abs/2005.14165)

[06] Kaplan, J., McCandlish, S., Henighan, T., Brown, T. B., Chess, B., Child, R., ... & Amodei, D. (2020).
**Scaling Laws for Neural Language Models.** arXiv.
[Paper](https://arxiv.org/abs/2001.08361)

[07] Raffel, C., Shazeer, N., Roberts, A., Lee, K., Narang, S., Matena, M., ... & Liu, P. J. (2020).
**Exploring the Limits of Transfer Learning with a Unified Text-to-Text Transformer (T5).** Journal of Machine Learning Research (JMLR).
[Paper](https://arxiv.org/abs/1910.10683)

[08] Chowdhery, A., Narang, S., Devlin, J., Bosma, M., Mishra, G., Roberts, A., ... & Dean, J. (2022).
**PaLM: Scaling Language Models with Pathways.** arXiv.
[Paper](https://arxiv.org/abs/2204.02311)

[09] Wei, J., Wang, X., Schuurmans, D., Bosma, M., Ichter, B., Xia, F., ... & Zhou, D. (2022).
**Chain-of-Thought Prompting Elicits Reasoning in Large Language Models.** Advances in Neural Information Processing Systems (NeurIPS).
[Paper](https://arxiv.org/abs/2201.11903)

[10] OpenAI (2023).
**GPT-4 Technical Report.** arXiv.
[Paper](https://arxiv.org/abs/2303.08774)

[11] Touvron, H., Lavril, T., Izacard, G., Martinet, X., Lachaux, M. A., Lacroix, T., ... & Lample, G. (2023).
**LLaMA: Open and Efficient Foundation Language Models.** arXiv.
[Paper](https://arxiv.org/abs/2302.13971)


----
### Questions

<div class="alert alert-block alert-info">Fill in your responses and save this notebook. See the <a href="../../03_Assignments/Submission-Guidelines.md">submission guidelines</a> for how to submit your work.</div>

**What core principle do LLMs use to generate text?** Answer:

(add your answer here)

**Experiment with the weights in our tiny model from Section 2. What do you notice?** Answer:

(add your answer here)

**In Section 2, which lookback window works better for this example? How does the demo behave if you adjust the weights of `cat`/`cats`?** Answer:

(add your answer here)

**Given the starting token `these`, weight w1 for `cat` and weight w2 for `cats`: what is the probability that the choice is correct? (answer this for both window sizes)** Answer:

(add your answer here)

**Why do you think LLM developers often do not disclose exact training details and parameters?** Answer:

(add your answer here)

**According to the “more data + more compute + larger models” view, why does scaling alone not always improve quality?** Answer:

(add your answer here)

**How would you set decoding parameters for a factual, deterministic task?** Answer:

(add your answer here)

**How would you set the decoding parameters for brainstorming or creative writing?** Answer:

(add your answer here)


**When running the parameter demos, what stood out? Which parameter had the biggest impact, and which had the least?** Answer:

(add your answer here)

**What would be a realistic scenario where you would use both top-k and top-p, together? What guardrail does top-k add to top-p?** Answer:

(add your answer here)

**OPTIONAL: Did you read through and understand all the tasks in this Notebook? Are there any shortcomings, missing parts, or sections that should be improved?**

 (please help the team by telling us what part was not understood, and we will work on it)